In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import torch.optim as optifrom
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [63]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Using Apple Silicon GPU acceleration (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 Using Nvidia GPU acceleration (CUDA)")
else:
    device = torch.device("cpu")
    print("⚠️ GPU backend not found, falling back to CPU")

🚀 Using Apple Silicon GPU acceleration (MPS)


In [64]:
# Jupyter cells do not define __file__; use the notebook kernel's working directory.
base_dir = os.getcwd()
csv_path = os.path.join(base_dir, 'Lorentz_attractor_dropped_10pct.csv')
df = pd.read_csv(csv_path)
df.head()

,Time_s,X,Y,Z,dX_dt,dY_dt,dZ_dt
0,0.00,1.000000,1.000000,1.000000,0.000000,26.000000,-1.666667
1,0.01,1.012502,1.260005,0.984891,2.475030,26.092840,-1.350618
2,0.02,1.048747,1.524097,0.973118,4.753492,26.820278,-0.996587
3,0.03,1.107219,1.798300,0.965171,6.910806,28.135175,-0.582677
4,0.04,1.186885,2.088526,0.961760,9.016412,30.002755,-0.085854


In [65]:
p = df[["Time_s","X","Y","Z"]]
p_t = df[["Time_s"]]
len(p)


1801

In [66]:
dt = df["Time_s"].diff().dropna().mean()
print(dt)

0.011111111111111112


In [70]:
def derivative(y,h):
    n = y.shape[0]
    dydt = np.zeros_like(y, dtype=np.float64)
    
    if n >= 5:
        dydt[2:n-2] = (-y[4:n] + 8*y[3:n-1] - 8*y[1:n-3] + y[0:n-4]) / (12.0 * h)
    
    if n >= 3:
        dydt[1] = (y[2] - y[0]) / (2.0 * h)
        dydt[n-2] = (y[n-1] - y[n-3]) / (2.0 * h)
    
    if n >= 3:
        dydt[0] = (-3*y[0] + 4*y[1] - y[2]) / (2.0 * h)
        dydt[n-1] = (3*y[n-1] - 4*y[n-2] + y[n-3]) / (2.0 * h)
    elif n == 2:
        dydt[0] = dydt[1] = (y[1] - y[0]) / h
    
    return dydt

computed_D = derivative(np.asarray(p, dtype=np.float64), dt)
